# Build Master Dataset

This notebook combines all sector/index CSVs and CBSL daily/monthly CSVs into one master research dataset.

Output file: `master_dataset.csv` in the same folder.

In [41]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

base_dir = Path(r"c:/Files/Uni/S4/DS/Project Dataset/Raw Data")
all_csvs = sorted(base_dir.glob("*.csv"))

all_csvs

[WindowsPath('c:/Files/Uni/S4/DS/Project Dataset/Raw Data/ASPI.csv'),
 WindowsPath('c:/Files/Uni/S4/DS/Project Dataset/Raw Data/Automobiles & Components.csv'),
 WindowsPath('c:/Files/Uni/S4/DS/Project Dataset/Raw Data/Banks.csv'),
 WindowsPath('c:/Files/Uni/S4/DS/Project Dataset/Raw Data/Capital Goods.csv'),
 WindowsPath('c:/Files/Uni/S4/DS/Project Dataset/Raw Data/CBSL Daily data.csv'),
 WindowsPath('c:/Files/Uni/S4/DS/Project Dataset/Raw Data/CBSL monthly data.csv'),
 WindowsPath('c:/Files/Uni/S4/DS/Project Dataset/Raw Data/Commercial & Professional services.csv'),
 WindowsPath('c:/Files/Uni/S4/DS/Project Dataset/Raw Data/Consumer Durables & Apparel.csv'),
 WindowsPath('c:/Files/Uni/S4/DS/Project Dataset/Raw Data/Consumer Services.csv'),
 WindowsPath('c:/Files/Uni/S4/DS/Project Dataset/Raw Data/Diversified Financials.csv'),
 WindowsPath('c:/Files/Uni/S4/DS/Project Dataset/Raw Data/Energy.csv'),
 WindowsPath('c:/Files/Uni/S4/DS/Project Dataset/Raw Data/Food & Staples Retailing.csv'),


In [42]:
# Files with daily sector/index series in the format: Traded Date, Value
series_files = [
    "ASPI.csv",
    "Banks.csv",
    "Capital Goods.csv",
    "Consumer Durables & Apparel.csv",
    "Consumer Services.csv",
    "Diversified Financials.csv",
    "Energy.csv",
    "Food, Beverage & Tobacco.csv",
    "Healthcare Equipment & Services.csv",
    "Insurance.csv",
    "Materials.csv",
    "Real Estate Management & Development.csv",
    "Retailing.csv",
    "Telecommunication Services.csv",
]

master = None

for name in series_files:
    file_path = base_dir / name
    df = pd.read_csv(file_path)
    df.columns = [c.strip() for c in df.columns]

    if "Traded Date" not in df.columns or "Value" not in df.columns:
        raise ValueError(f"Unexpected columns in {name}: {df.columns.tolist()}")

    out_col = Path(name).stem.replace(" ", "_").replace("&", "and")

    s = df[["Traded Date", "Value"]].copy()
    s["date"] = pd.to_datetime(s["Traded Date"], format="%d %b %Y", errors="coerce")
    s[out_col] = pd.to_numeric(s["Value"], errors="coerce")
    s = s[["date", out_col]].dropna(subset=["date"]).sort_values("date")

    if master is None:
        master = s
    else:
        master = master.merge(s, on="date", how="outer")

master = master.sort_values("date").reset_index(drop=True)
master.head()

,date,ASPI,Banks,Capital_Goods,Consumer_Durables_and_Apparel,Consumer_Services,Diversified_Financials,Energy,"Food,_Beverage_and_Tobacco",Healthcare_Equipment_and_Services,Insurance,Materials,Real_Estate_Management_and_Development,Retailing,Telecommunication_Services
0,2019-01-02,6062.20,826.06,851.86,789.61,253.78,767.38,617.20,946.43,863.13,2181.00,540.58,755.29,790.20,748.46
1,2019-01-03,6058.48,820.19,848.10,797.83,256.55,781.43,613.47,946.08,865.17,2166.21,542.73,749.56,787.44,740.51
2,2019-01-04,6067.66,823.60,850.90,812.56,255.14,778.30,603.23,948.34,866.56,2166.73,544.92,749.69,791.14,744.21
3,2019-01-07,6022.99,816.66,838.41,805.63,254.43,773.72,591.92,942.37,861.18,2137.48,540.84,746.16,782.55,744.21
4,2019-01-08,5992.36,808.46,835.02,802.10,252.63,773.00,585.58,937.21,859.54,2129.06,539.51,749.48,764.40,735.24


In [43]:
master.isna().sum()

date                                      0
ASPI                                      0
Banks                                     0
Capital_Goods                             0
Consumer_Durables_and_Apparel             0
Consumer_Services                         0
Diversified_Financials                    0
Energy                                    0
Food,_Beverage_and_Tobacco                0
Healthcare_Equipment_and_Services         0
Insurance                                 0
Materials                                 0
Real_Estate_Management_and_Development    0
Retailing                                 1
Telecommunication_Services                1
dtype: int64

In [44]:
# Forward-fill selected sector columns
cols_to_ffill = ["Retailing", "Telecommunication_Services"]

missing_cols = [c for c in cols_to_ffill if c not in master.columns]
if missing_cols:
    raise KeyError(f"Columns not found in master: {missing_cols}")

master[cols_to_ffill] = master[cols_to_ffill].ffill()

# Check remaining NaNs in the selected columns
master[cols_to_ffill].isna().sum()

Retailing                     0
Telecommunication_Services    0
dtype: int64

In [45]:
def reshape_cbsl_daily(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]

    target_items = [
        "Reverse Repo/Standing Lending Facility - Rate",
        "Repo/Standing Deposit Facility - Rate",
    ]

    date_cols = [c for c in df.columns if re.fullmatch(r"\d{2}/\d{2}/\d{4}", c)]

    use = df[df["Item Name"].astype(str).str.strip().isin(target_items)][["Item Name"] + date_cols].copy()

    long_df = use.melt(id_vars="Item Name", value_vars=date_cols, var_name="date", value_name="value")
    long_df["date"] = pd.to_datetime(long_df["date"], dayfirst=True, errors="coerce")
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df = long_df.dropna(subset=["date"])

    wide = long_df.pivot_table(index="date", columns="Item Name", values="value", aggfunc="last")
    wide.columns = [
        re.sub(r"[^0-9a-zA-Z]+", "_", c).strip("_")[:120]
        for c in wide.columns.astype(str)
    ]

    return wide.reset_index()


def reshape_cbsl_monthly(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]

    month_cols = [c for c in df.columns if re.fullmatch(r"\d{4}-[A-Za-z]{3}", c)]

    use = df[["Item Name"] + month_cols].copy()

    target_items = [
        "CCPI(2021=100)(Headline)",
        "CCPI(2013=100)(Headline)",
        "Monthly Average Exchange Rates",
    ]
    use = use[use["Item Name"].astype(str).str.strip().isin(target_items)].copy()

    long_df = use.melt(id_vars="Item Name", value_vars=month_cols, var_name="month", value_name="value")
    long_df["month"] = pd.to_datetime(long_df["month"], format="%Y-%b", errors="coerce")
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df = long_df.dropna(subset=["month"])

    wide = long_df.pivot_table(index="month", columns="Item Name", values="value", aggfunc="last")
    wide.columns = [
        re.sub(r"[^0-9a-zA-Z]+", "_", c).strip("_")[:120]
        for c in wide.columns.astype(str)
    ]

    return wide.reset_index()


CBSL_daily_path = base_dir / "CBSL Daily data.csv"
CBSL_monthly_path = base_dir / "CBSL monthly data.csv"

# Transform CBSL daily and monthly files into merge-ready tables.
cbsl_daily = reshape_cbsl_daily(CBSL_daily_path)
cbsl_monthly = reshape_cbsl_monthly(CBSL_monthly_path)

# Merge daily CBSL indicators directly using exact calendar date.
if not cbsl_daily.empty:
    master = master.merge(cbsl_daily, on="date", how="left")

# Merge monthly CBSL indicators by creating a month bucket from each daily date.
if not cbsl_monthly.empty:
    # Convert each daily date to the first day of its month (timestamp).
    master["month"] = master["date"].dt.to_period("M").dt.to_timestamp()
    # Join monthly indicators using the month key.
    master = master.merge(cbsl_monthly, on="month", how="left")
    # Remove temporary month key after merge.
    master = master.drop(columns=["month"])

# Final cleanup: sort by date and reset row index.
master = master.sort_values("date").reset_index(drop=True)
master.head()

,date,ASPI,Banks,Capital_Goods,Consumer_Durables_and_Apparel,Consumer_Services,Diversified_Financials,Energy,"Food,_Beverage_and_Tobacco",Healthcare_Equipment_and_Services,Insurance,Materials,Real_Estate_Management_and_Development,Retailing,Telecommunication_Services,Repo_Standing_Deposit_Facility_Rate,Reverse_Repo_Standing_Lending_Facility_Rate,CCPI_2013_100_Headline,CCPI_2021_100_Headline,Monthly_Average_Exchange_Rates
0,2019-01-02,6062.20,826.06,851.86,789.61,253.78,767.38,617.20,946.43,863.13,2181.00,540.58,755.29,790.20,748.46,8.0,9.0,127.4,NaN,182.13
1,2019-01-03,6058.48,820.19,848.10,797.83,256.55,781.43,613.47,946.08,865.17,2166.21,542.73,749.56,787.44,740.51,8.0,9.0,127.4,NaN,182.13
2,2019-01-04,6067.66,823.60,850.90,812.56,255.14,778.30,603.23,948.34,866.56,2166.73,544.92,749.69,791.14,744.21,8.0,9.0,127.4,NaN,182.13
3,2019-01-07,6022.99,816.66,838.41,805.63,254.43,773.72,591.92,942.37,861.18,2137.48,540.84,746.16,782.55,744.21,8.0,9.0,127.4,NaN,182.13
4,2019-01-08,5992.36,808.46,835.02,802.10,252.63,773.00,585.58,937.21,859.54,2129.06,539.51,749.48,764.40,735.24,8.0,9.0,127.4,NaN,182.13


In [46]:
master.isna().sum()

date                                             0
ASPI                                             0
Banks                                            0
Capital_Goods                                    0
Consumer_Durables_and_Apparel                    0
Consumer_Services                                0
Diversified_Financials                           0
Energy                                           0
Food,_Beverage_and_Tobacco                       0
Healthcare_Equipment_and_Services                0
Insurance                                        0
Materials                                        0
Real_Estate_Management_and_Development           0
Retailing                                        0
Telecommunication_Services                       0
Repo_Standing_Deposit_Facility_Rate              1
Reverse_Repo_Standing_Lending_Facility_Rate      1
CCPI_2013_100_Headline                         700
CCPI_2021_100_Headline                         690
Monthly_Average_Exchange_Rates 

In [47]:
cutoff_date = pd.Timestamp("2025-08-31")
master = master[master["date"] <= cutoff_date].copy()

deposit_col = "Repo_Standing_Deposit_Facility_Rate"
lending_col = "Reverse_Repo_Standing_Lending_Facility_Rate"
for col in [deposit_col, lending_col]:
    if col in master.columns:
        master[col] = master[col].ffill()

ccpi_2013_col = "CCPI_2013_100_Headline"
ccpi_2021_col = "CCPI_2021_100_Headline"

monthly_ccpi = master[["date", ccpi_2013_col, ccpi_2021_col]].copy()
monthly_ccpi["month"] = monthly_ccpi["date"].dt.to_period("M").dt.to_timestamp()
monthly_ccpi = monthly_ccpi.sort_values("date").groupby("month", as_index=False).last()

overlap_start = pd.Timestamp("2022-01-01")
overlap_end = pd.Timestamp("2023-01-01")
overlap = monthly_ccpi[
    (monthly_ccpi["month"] >= overlap_start)
    & (monthly_ccpi["month"] <= overlap_end)
    & monthly_ccpi[ccpi_2013_col].notna()
    & monthly_ccpi[ccpi_2021_col].notna()
].copy()

ratio_series = overlap[ccpi_2021_col] / overlap[ccpi_2013_col].replace(0, np.nan)
avg_ratio_2021_over_2013 = ratio_series.mean()

master["CCPI_2013_Rebased_to_2021"] = master[ccpi_2013_col] * avg_ratio_2021_over_2013

master["CCPI_2021_Base"] = master[ccpi_2021_col].combine_first(master["CCPI_2013_Rebased_to_2021"]).round(1)

master.drop(columns=[ccpi_2013_col, ccpi_2021_col,"CCPI_2013_Rebased_to_2021"], inplace=True)

master.isna().sum()

date                                           0
ASPI                                           0
Banks                                          0
Capital_Goods                                  0
Consumer_Durables_and_Apparel                  0
Consumer_Services                              0
Diversified_Financials                         0
Energy                                         0
Food,_Beverage_and_Tobacco                     0
Healthcare_Equipment_and_Services              0
Insurance                                      0
Materials                                      0
Real_Estate_Management_and_Development         0
Retailing                                      0
Telecommunication_Services                     0
Repo_Standing_Deposit_Facility_Rate            0
Reverse_Repo_Standing_Lending_Facility_Rate    0
Monthly_Average_Exchange_Rates                 0
CCPI_2021_Base                                 0
dtype: int64

In [48]:
output_path = base_dir / "master_dataset.csv"
master.to_csv(output_path, index=False)

master.tail()

,date,ASPI,Banks,Capital_Goods,Consumer_Durables_and_Apparel,Consumer_Services,Diversified_Financials,Energy,"Food,_Beverage_and_Tobacco",Healthcare_Equipment_and_Services,Insurance,Materials,Real_Estate_Management_and_Development,Retailing,Telecommunication_Services,Repo_Standing_Deposit_Facility_Rate,Reverse_Repo_Standing_Lending_Facility_Rate,Monthly_Average_Exchange_Rates,CCPI_2021_Base
1554,2025-08-25,20575.59,1745.38,2262.44,2110.26,625.28,4001.98,2619.26,2162.43,1575.38,2468.68,2752.17,2252.17,2633.61,1932.36,7.25,8.25,301.12,193.3
1555,2025-08-26,20613.39,1752.92,2262.87,2130.07,627.74,4024.32,2602.81,2162.41,1560.92,2457.40,2782.90,2239.68,2631.95,1972.65,7.25,8.25,301.12,193.3
1556,2025-08-27,20753.21,1753.80,2259.39,2171.67,635.69,4033.37,2615.18,2191.19,1569.02,2438.06,2809.76,2296.76,2708.29,1993.87,7.25,8.25,301.12,193.3
1557,2025-08-28,20800.26,1748.18,2258.29,2252.89,634.43,4042.59,2634.75,2202.09,1587.52,2452.28,2840.82,2371.70,2734.00,2070.86,7.25,8.25,301.12,193.3
1558,2025-08-29,20997.36,1753.04,2270.74,2268.65,649.72,4074.07,2658.56,2219.24,1606.10,2524.84,2871.93,2517.90,2750.54,2173.82,7.25,8.25,301.12,193.3
